Firstly,
1. Download nltk libraries
2. download spacy libraries

In [4]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\visha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\visha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### News Article Analsis:
A Complete Pipeline project which would help user get insights on any news article.

In [5]:
import numpy as np
import pandas as pd

#### 1. BBC News
This dataset contains news article from bbc news. Each article is also categorized in a single category based on its text.

#### 2. News Aggregation with summary from Narayan, Shashi, Shay B. Cohen, and Mirella Lapata. (2018).
This dataset contains news article from various news sources. Each article also have a human generated summary along with unique ID. Original dataset has 580014 article but that is way too large for our project. <br>
So, we would take a fraction of complete dataset.
<link>[text](https://www.kaggle.com/datasets/sbhatti/news-summarization)</link>

In [6]:
# Importing dataset
data = pd.read_csv("../Dataset/bbc-text.csv")
print("Shape: ",data.shape)
print("Five random sample points:")
data.sample(5)

## Main dataset for project
df = pd.read_csv("../Dataset/data_sample.csv", index_col=0)
df

Shape:  (2225, 2)
Five random sample points:


,ID,Content,Summary,Dataset
0,f49ee725a0360aa6881ed1f7999cc531885dd06a,New York police are concerned drones could bec...,Police have investigated criminals who have ri...,CNN/Daily Mail
1,808fe317a53fbd3130c9b7563341a7eea6d15e94,By . Ryan Lipman . Perhaps Australian porn sta...,Porn star Angela White secretly filmed sex act...,CNN/Daily Mail
2,98fd67bd343e58bc4e275bbb5a4ea454ec827c0d,"This was, Sergio Garcia conceded, much like be...",American draws inspiration from fellow country...,CNN/Daily Mail
3,e12b5bd7056287049d9ec98e41dbb287bd19a981,An Ebola outbreak that began in Guinea four mo...,World Health Organisation: 635 infections and ...,CNN/Daily Mail
4,b83e8bcfcd51419849160e789b6658b21a9aedcd,By . Associated Press and Daily Mail Reporter ...,A sinkhole opened up at 5:15am this morning in...,CNN/Daily Mail
...,...,...,...,...
9995,39372214,Os na fydd modd dod o hyd i berchennog bydd y ...,Mae cwmni Tindle Newspapers wedi cadarnhau ei ...,XSum
9996,5bd64292bb2a7260c6c3d1156c6a03186b4699ba,Aston Villa have made a late charge to steal M...,Newcastle have made Dele Alli a primary transf...,CNN/Daily Mail
9997,877537a4a49d9ac48d1850e4ca7a83278d6487a9,"By . Nick Mcdermott, Science Reporter . PUBLIS...",National Childbirth Trust accused of championi...,CNN/Daily Mail
9998,021ea5c8d96f0466c810ec304059095437886dab,(CNN) -- In anticipation of more flooding next...,Fargo spokeswoman says city has goal of fillin...,CNN/Daily Mail


In [7]:
# # Importing dataset
# df = pd.read_excel("../Dataset/MN-DS-news-classification.xlsx")
# # Identify columns to drop
# unnamed_cols = [col for col in df.columns if 'Unnamed' in col]

# df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
# print("Shape: ",df.shape)
# print("Three random sample points:")
# df.sample(3)

In [8]:
# # Converting date into valid readable format
# df['date'] = pd.to_numeric(df['date'], errors='coerce')
# df['date'] = pd.to_datetime(df['date'], unit='D', origin='1899-12-30')

### Now, we will analyse our dataset

First, Lets have a High level overview using Automatic EDA.

In [9]:
# import dtale

# rep = dtale.show(df)
# rep

#### Step-1: High Level Overview

In [10]:
# 1) Basic info and structure:
print("List of all columns: ",df.columns)
print("Column wise info: ",df.info())

List of all columns:  Index(['ID', 'Content', 'Summary', 'Dataset'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   ID       9357 non-null   str  
 1   Content  10000 non-null  str  
 2   Summary  10000 non-null  str  
 3   Dataset  10000 non-null  str  
dtypes: str(4)
memory usage: 43.3 MB
Column wise info:  None


In [11]:
# 2) Datatype for each columns -> object
print("Datatype of each column: ",df.dtypes)

Datatype of each column:  ID         str
Content    str
Summary    str
Dataset    str
dtype: object


In [12]:
# 3) Duplicates -> Remove them all
print("Number of duplicate rows in main dataset: ",df[df.duplicated()].shape[0])
print("Percentage of duplicate rows in main dataset: ",df[df.duplicated()].shape[0]/df.shape[0]*100)
dup_index = df.index[df.duplicated(keep='first')]
# df_clean = df.drop(dup_index)

## Similarily for bbc dataset
print("Number of duplicate rows in BBC dataset: ",data[data.duplicated()].shape[0])
print("Percentage of duplicate rows in BBC dataset: ",data[data.duplicated()].shape[0]/data.shape[0]*100)

Number of duplicate rows in main dataset:  37
Percentage of duplicate rows in main dataset:  0.37
Number of duplicate rows in BBC dataset:  99
Percentage of duplicate rows in BBC dataset:  4.449438202247191


In [13]:
# 4) Missing values
missing_count = df.isnull().sum()
print("Missing Count:\n",missing_count)

missing_dataset = df[df.isnull().any(axis=1)]
print("\nNumber of rows with missing values: ",missing_dataset.shape[0])
print("\nPercentage of missing values: \n",missing_count/df.shape[0]*100)

Missing Count:
 ID         643
Content      0
Summary      0
Dataset      0
dtype: int64

Number of rows with missing values:  643

Percentage of missing values: 
 ID         6.43
Content    0.00
Summary    0.00
Dataset    0.00
dtype: float64


In [14]:
## Similarily for bbc dataset
missing_count = data.isnull().sum()
missing_dataset = data[data.isnull().any(axis=1)]
print("Number of rows with missing values: ",missing_dataset.shape[0])

Number of rows with missing values:  0


Now, we will clean our dataset based off problem from high-level overview. <br>
-> Missing values + Duplicates + Textual problems + Type conversion

1. There are some article with no ID. As we have no need for any ID, we can just ignore these missing values.
2. As their are few duplicates, we can just remove them from our dataset. (dup < 5% of total dataset)
3. There are only three 3 defined source, we can convert this into category.
4. Rows with missing content are useless. Remove them from dataset.

In [15]:
# looking into dataset source
print(df['Dataset'].value_counts())

# Since their are only three source, we can convert this into category
df['Dataset'] = df['Dataset'].astype('category')

# Removing duplicate
shape_before = df.shape
df = df.drop_duplicates(subset=['ID','Content'], keep='first')
print("Number of rows after removing duplicates: ",-df.shape[0] + shape_before[0])
shape_before = data.shape
data = data.drop_duplicates()
print("Number of rows after removing duplicates: ",-data.shape[0] + shape_before[0])

## Removing missing values
shape_before = df.shape
df = df.dropna(subset=['Content'])
print("Number of rows after removing missing values: ",-df.shape[0] + shape_before[0])

Dataset
CNN/Daily Mail    6787
XSum              2570
Multi-News         643
Name: count, dtype: int64
Number of rows after removing duplicates:  37
Number of rows after removing duplicates:  99
Number of rows after removing missing values:  0


In [16]:
## Removing unicodes:
import unicodedata, re
import html
from bs4 import BeautifulSoup

def clean_text(s: str) -> str:
    """
    Perform safe, canonical text cleaning for NLP tasks.
    Preserves linguistic structure.
    """
    if pd.isna(s) or not isinstance(s, str):
        return ''
    
    # 1. Fix broken encoding and HTML entities
    s = html.unescape(s)    
    # 2. normalize unicode (NFKC helps)
    s = unicodedata.normalize('NFKC', s)
    # 3. Remove HTML tags (robust)
    s = BeautifulSoup(s, 'lxml').get_text(separator=" ")
    # 4. remove ZERO WIDTH and BOM chars
    s = re.sub(r'[\u200B-\u200D\uFEFF]', '', s)
    # 5. Normalize whitespace (spaces, tabs)
    s = re.sub(r"[ \t]+", " ", s)
    # 6. Remove repeated newlines
    s = re.sub(r"\n\s*\n+", "\n", s)
    # 7. Strip leading and trailing whitespace
    s = s.strip()
    
    return s


In [17]:
# Create a cleaned column for diagnostics
df['Content'] = df['Content'].apply(clean_text)
df['Summary'] = df['Summary'].apply(clean_text)
data['text'] = data['text'].apply(clean_text)

In [18]:
## Now, lets save these cleaned datasets
df.to_parquet('../Dataset/Clean/Clean_dataset.parquet')
data.to_parquet('../Dataset/Clean/Clean_bbc.parquet')

At this point we would look at other data cleaning problems like
* Spelling correction
* Outliers
* Data Transformation
* Imbalance for supervised task

But, 
- Our dataset is already checked for spelling mistake.
- There is no target label for Imbalance
- There are no outliers due to no numerical features yet
- There are no hardcore direct transformations required for complete raw text.

Now, We will process our textual data and create new features.

##### 🧱 A. Length & Size Features (FOUNDATIONAL)
These are always useful.
- char_count
- char_count_no_spaces
- word_count
- unique_word_count
- sentence_count

📌 Used in:
1. EDA
2. Quality checks
3. Readability
4. Feature dashboards

##### 🧠 B. Vocabulary & Lexical Richness
Measures how diverse the language is.
- lexical_diversity = unique_words / total_words
- hapax_legomena_ratio (words appearing once)
- top_k_word_ratio (dominance of common words)

📌 Useful for:
1. Article style analysis
2. Fake/low-quality detection
3. Comparing news outlets

##### 🚦 C. Stopword & Function Word Features
Stopwords are signal, not noise, for analysis.
- stopword_count
- stopword_ratio
- content_word_count
- content_word_ratio

📌 Useful for:
1. Readability
2. Writing style
3. Compression difficulty (summarization)

##### 🧾 D. Punctuation & Formatting Features [Not useful for our tasks]
Reflect writing structure.
- comma_count
- period_count
- question_mark_count
- exclamation_count
- quote_count
- punctuation_density

📌 Useful for:
1. Narrative vs factual tone
2. Opinion vs reporting

##### 🧠 E. POS (Part-of-Speech) Statistics
Very powerful, often ignored.
- noun_count
- verb_count
- adj_count
- adv_count
- pronoun_count
- pos_distribution

📌 Useful for:
1. Topic style
2. Abstraction level
3. Keyword extraction quality

##### 🧍 F. Named Entity Statistics
Blind, no labels required.
- person_count
- org_count
- gpe_count
- event_count
- unique_entity_count

📌 Useful for:
1. Newsworthiness
2. Topic richness
3. Similarity explanations

##### 📖 G. Readability & Complexity [Advance]
Classic but still relevant.
- flesch_reading_ease == Rate text reading ease [0-100]
- flesch_kincaid_grade == approximate the US grade level required to understand the text
- gunning_fog == Indicate years of formal education needed to understand the text

📌 Useful for:
1. User-facing insights
2. Comparing summaries vs originals

##### 🔁 H. Repetition & Redundancy [Not much useful]
- repeated_sentence_ratio
- repeated_word_ratio

📌 Useful for:
1. Detecting low-quality content
2. Summarization difficulty

In [19]:
import spacy
import textstat
from collections import Counter

nlp = spacy.load('en_core_web_sm')

df = pd.read_parquet('../Dataset/Clean/Clean_dataset.parquet')
data = pd.read_parquet('../Dataset/Clean/Clean_bbc.parquet')

d:\Krishan\Project dataset\News categorization\.NewsEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
## Completely spacy pipeline
def Extract_text_features(text: str) -> dict:
    doc = nlp(text)

    tokens = [t for t in doc if not t.is_space]
    words = [t for t in tokens if t.is_alpha]

    word_texts = [t.text.lower() for t in words]
    unique_words = set(word_texts)

    stopwords = [t for t in words if t.is_stop]
    content_words = [t for t in words if not t.is_stop]

    pos_counts = Counter(t.pos_ for t in words)
    ent_counts = Counter(ent.label_ for ent in doc.ents)

    features = {
        # Length
        "char_count": len(text),
        "char_count_no_spaces": len(text.replace(" ", "")),
        "word_count": len(words),
        "unique_word_count": len(unique_words),
        "sentence_count": len(list(doc.sents)),

        # Lexical
        "lexical_diversity": len(unique_words) / max(len(words), 1),
        "avg_word_length": sum(len(w) for w in word_texts) / max(len(words), 1),
        "avg_sentence_length": len(words) / max(len(list(doc.sents)), 1),

        # Stopwords
        "stopword_count": len(stopwords),
        "stopword_ratio": len(stopwords) / max(len(words), 1),
        "content_word_count": len(content_words),

        # POS
        "noun_count": pos_counts.get("NOUN", 0),
        "verb_count": pos_counts.get("VERB", 0),
        "adj_count": pos_counts.get("ADJ", 0),
        "adv_count": pos_counts.get("ADV", 0),
        "pronoun_count": pos_counts.get("PRON", 0),

        # Entities
        "person_count": ent_counts.get("PERSON", 0),
        "org_count": ent_counts.get("ORG", 0),
        "gpe_count": ent_counts.get("GPE", 0),
        "event_count": ent_counts.get("EVENT", 0),
        "unique_entity_count": len(set(ent.text for ent in doc.ents)),

        # Readability
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
        "gunning_fog": textstat.gunning_fog(text),
    }

    return features


Spacy is an easy to use library but it does not perform all these feature accurately. <br>
So, we will use other libraries to extract features.

In [21]:
# NLP libs
import nltk

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from flair.models import SequenceTagger
from flair.data import Sentence

> Some features are not code due to correlation b/w independent variable aka Multicollinearity

In [22]:
## 1. Text Length & Structure
def length_features(text:str) -> dict:
    sentences = sent_tokenize(text)
    words = [w for w in word_tokenize(text) if w.isalpha()]
    
    return {
        "char_count": len(text),
        "char_count_no_spaces": len(text.replace(" ", "")),
        "sentence_count": len(sentences),
    }

In [23]:
## 2. Vocabulary & Lexical Richness [NLTK]
nltk.download('punkt')

def lexical_features(text:str) -> dict:
    tokens = [t.lower() for t in word_tokenize(text) if t.isalpha()]
    counts = Counter(tokens)
    
    rare_words = [w for w,c in counts.items() if c==1]
    
    return {
        "word_count": len(tokens),
        "unique_word_count": len(counts),
        "lexical_diversity": len(counts) / max(len(tokens), 1),
        # ratio of single occuring word/total words
        "hapax_ratio": len(rare_words) / max(len(tokens), 1)
    }

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\visha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [24]:
## 3. Stopwords and Content-Word Features [NLTK]
nltk.download('stopwords')

stopword_list = set(stopwords.words('english'))

def stopword_features(text:str) -> dict:
    tokens = [t for t in word_tokenize(text) if t.isalnum()]
    stopwords_used = [t for t in tokens if t in stopword_list]
    
    return {
        "stopword_count": len(stopwords_used),
        "content_word_count": len(tokens) - len(stopwords_used),
        "stopword_ratio": len(stopwords_used) / max(1, len(tokens)),
    }

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\visha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [25]:
## 4. POS Statistics [SpaCy]
def pos_features(text:str) -> dict:
    doc = nlp(text)
    pos_counts = Counter(tok.pos_ for tok in doc if tok.is_alpha)
    
    return {
        "noun_count":pos_counts.get("NOUN", 0),
        "verb_count":pos_counts.get("VERB", 0),
        "adj_count":pos_counts.get("ADJ", 0),
        "adv_count":pos_counts.get("ADV", 0),
        "pronoun_count":pos_counts.get("PRON", 0),
    }

In [26]:
## 5. NER features [Spacy]
def ner_features(text:str) -> dict:
    doc = nlp(text)
    ent_counts = Counter(ent.label_ for ent in doc.ents)
    
    return {
        "person_count": ent_counts.get("PERSON", 0),
        "org_count": ent_counts.get("ORG", 0),
        "gpe_count": ent_counts.get("GPE", 0),
        "event_count": ent_counts.get("EVENT", 0),
        "unique_entity_count": len(set(ent.text for ent in doc.ents))
    }

tagger = SequenceTagger.load("flair/ner-english")

## 5. NER strong [Flair]
def _ner_features(text:str) -> dict:
    sentence = Sentence(text)
    tagger.predict(sentence)

    entities = [ent.get_label("ner").value for ent in sentence.get_spans("ner")]
    ent_counts = Counter(entities)

    return {
        "person_count": ent_counts.get("PER", 0),
        "org_count": ent_counts.get("ORG", 0),
        "location_count": ent_counts.get("LOC", 0),
        "misc_entity_count": ent_counts.get("MISC", 0),
        "total_entity_count": len(entities)
    }

2026-02-20 16:07:52,976 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>


FLAIR is a simple and powerful Python library for NLP. It helps perform task like NER, POS, Chunking, Relation extraction, sentiment analysis, etc. <br>

<b> Key Features :</b>
- Contextual String Embeddings - uses character level language model to produce context sensitive word representations
- Stackable Embeddings - combine embeddings like  GloVe, BERT, ELMo, etc.
- Pre-trained Language Models - uses pretrained language models to do anything you want
- Easy Model training and Fine tuning

In [27]:
## 6. Readability Features [Textstat]
def readability_features(text:str) -> dict:
    return {
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
        "gunning_fog": textstat.gunning_fog(text)
    }

In [28]:
# Applying all on data
def extract_text_features(text:str) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {}

    features = {}

    # 1. Length & structure
    features.update(length_features(text))
    # 2. Lexical richness
    features.update(lexical_features(text))
    # 3. Stopwords
    features.update(stopword_features(text))
    # 4. POS statistics (spaCy)
    features.update(pos_features(text))
    # 5. NER (choose ONE – Flair preferred for performance but spacy for speed)
    # features.update(_ner_features(text))   # stronger NER
    features.update(ner_features(text))  # spaCy alternative
    # 6. Readability
    features.update(readability_features(text))

    return features

In [29]:
from tqdm import tqdm

def add_features_to_dataframe(DF:pd.DataFrame, text_col:str):
    """
    Apply NLP feature extractio to each row of any dataframe
    """
    if text_col not in DF.columns:
        raise ValueError(f"Column '{text_col}' not found in dataframe")
    
    texts = DF[text_col].fillna("").astype(str)
    features = texts.apply(extract_text_features)
    
    # Merge
    features_df = pd.DataFrame(features.tolist(), index=DF.index)
    df_enriched = pd.concat([DF, features_df], axis=1)
    
    return df_enriched

In [30]:
group = data.groupby("text").count().sort_values("category", ascending=False)
group.head(2)

,category
text,
davenport hits out at wimbledon world number one lindsay davenport has criticised wimbledon over the issue of equal prize money for women. reacting to a disputed comment by all england club chairman tim phillips the american said: i think it is highly insulting if prize money is taken away. somebody i think it was mr phillips said they won t have money for flowers at wimbledon. that s insulting. an all england club spokesperson denied phillips made the remark insisting: he definitely didn t say it. the statement added: it was said by someone else and was a humorous aside at the end of a radio interview when the conversation had moved to talking about the wimbledon grounds. davenport was speaking following the announcement that this week s dubai duty free event will join the us and australian opens in offering equal prize money for women. you hear about women playing only three sets while men play five said daveport. and the best women are never going to beat the best men. but it s a different game you go to watch with the women - it doesn t make it better or worse. hopefully we will be able to change people s minds. serena williams who is also in dubai added: i m obviously for equal prize money. women s tennis is exciting. men s tennis is exciting as well but the women have it right now. if you are bringing in the spectators you should be able to reap what everyone else is able to reap.,2
desailly backs blues revenge trip marcel desailly insists there is no chance of history repeating itself when chelsea take on barcelona on wednesday. the french star was part of the chelsea side crushed 5-1 at the nou camp in the champions league quarter-final second leg in 2000. things will be totally different this time he told bbc sport. now everyone knows about chelsea and is a little bit afraid of them. they are one of the major clubs in europe and the pressure will be on barcelona. chelsea have not played barcelona since that quarter-final tie five years ago. the blues had looked destined to progress after winning the first leg at stamford bridge 3-1 courtesy of two goals from tore andre flo and one by gianfranco zola. but they collapsed in the second leg going down to strikes from rivaldo (2) luis figo dani and patrick kluivert. former chelsea captain desailly who is now playing for al-gharafa in qatar says there is no comparison between that side and the current blues team who are top of the premiership. mentally they are much stronger even though a lot of their players are young the 36-year-old said. we made some mistakes at the nou camp in 2000 - a lot of them were individual mistakes. it would not happen now. this team has a new motivation and a different mentality. world cup winner desailly saw huge changes during his time at stamford bridge. he was signed for £4.6m from ac milan in 1998 by ruud gullit and went on to play under gianluca vialli and claudio ranieri. but the biggest change occurred when billionaire roman abramovich bought the club in 2003. desailly says the russian s arrival helped to instil a winning mentality at the club as well as a demand for success. the whole of chelsea is different now - the chairman the manager and all the players he said. everything is new and there is a huge determination to win. since that game in 2000 chelsea have gained more experience in europe and were very close to reaching the champions league final last season. desailly is one of the most decorated players in the history of football. he won the 1998 world cup and 2000 european championship with france the champions league in 1993 with marseilles and 1994 with ac milan two serie a titles and the fa cup in 2000 with chelsea. he is now winding down his career in qatar alongside the likes of frank lebeouf josep guardiola titi camara gabriel batistuta and christophe dugarry. so he is full of admiration for two of his colleagues from the great milan side of the mid-90s who are likely to line up against manchester united on

In [31]:
## Getting features from data
# data_enrich = add_features_to_dataframe(data, 'text')
# data_enrich.sample(3)

In [32]:
## Getting features from df
df_enriched = add_features_to_dataframe(df, 'Content')
df_enriched.sample(3)

,ID,Content,Summary,Dataset,char_count,char_count_no_spaces,sentence_count,word_count,unique_word_count,lexical_diversity,...,adv_count,pronoun_count,person_count,org_count,gpe_count,event_count,unique_entity_count,flesch_reading_ease,flesch_kincaid_grade,gunning_fog
2698,c99ca0723bfce86cb77fadb49b2793d1a3aa8131,A Tory minister's attack on a 'wealthy metropo...,James Brokenshire said well-off benefit from c...,CNN/Daily Mail,6225,5189,50,989,421,0.425683,...,44,79,19,12,6,0,38,50.818229,11.352701,14.135421
1146,848d6810d0435d6c8712a06e0779559302b01f3a,"Atlanta, Georgia (CNN) -- Atlanta's public sch...",Accrediting agency on Tuesday announced probat...,CNN/Daily Mail,4253,3592,22,646,300,0.464396,...,16,27,8,16,8,0,35,31.660395,16.262298,19.190508
4070,f6f33423907bd30f181295daa919cc6eb145768e,"By . Helen Lawson . PUBLISHED: . 08:55 EST, 30...",Matthew Picton's maps go beyond a simple A-Z l...,CNN/Daily Mail,6298,5171,57,1052,382,0.363118,...,14,32,31,30,42,5,93,59.879482,10.197395,11.648402


In [35]:
## Getting features for summary column
df_summary_enrich = add_features_to_dataframe(df, 'Summary')
df_summary_enrich.sample(3)

,ID,Content,Summary,Dataset,char_count,char_count_no_spaces,sentence_count,word_count,unique_word_count,lexical_diversity,...,adv_count,pronoun_count,person_count,org_count,gpe_count,event_count,unique_entity_count,flesch_reading_ease,flesch_kincaid_grade,gunning_fog
6962,c696c63b1cd516a745b45e07506ffe760b65c741,Council officers have ticked off the Sandringh...,Sandringham estate was selling pâté which brea...,CNN/Daily Mail,216,185,3,31,27,0.870968,...,0,1,1,1,0,0,2,55.889583,8.113750,9.266667
5142,17fafe996f475f265b61879f3f22f21f8ac690d5,A massive moth measuring a foot from wing tip ...,Animal usually found in Malaysia .\nExperts be...,CNN/Daily Mail,98,83,2,15,15,1.000000,...,1,1,0,0,1,0,1,41.302500,9.361667,11.000000
6788,bdbd31f36ac489bea8e4183cd88603665f80d981,"By . Emma Innes . PUBLISHED: . 12:11 EST, 13 J...","Methylisothiazolinon (MI), a preservative, can...",CNN/Daily Mail,329,276,4,53,46,0.867925,...,1,3,0,0,0,0,0,59.303231,8.279387,11.337736


In [36]:
## Saving the dataset
# df_enriched.to_parquet('../Dataset/Clean/dataset_with_features.parquet')
# data_enrich.to_parquet('../Dataset/Clean/BBC_with_features_combined.parquet')
df_summary_enrich.to_parquet('../Dataset/Clean/dataset_summary_features.parquet')